# Smart PDF Figure Screenshot Extractor

This notebook takes **screenshots of FIGURE regions** from PDFs, preserving:
- ✅ The actual diagram/image
- ✅ All text labels and annotations within the figure
- ✅ The complete figure caption below
- ✅ Figure numbers (FIGURE 1-1, FIGURE 2-1, etc.)

**This approach captures the entire figure as it appears in the book, not just the embedded image!**

Perfect for medical textbooks, anatomy diagrams, and any figures with important text annotations!

## Step 1: Install Required Libraries

In [ ]:
# Install required packages
!pip install -q pymupdf PyMuPDF Pillow

print("✅ All packages installed successfully!")

## Step 2: Import Libraries

In [ ]:
import fitz  # PyMuPDF
import os
from pathlib import Path
from PIL import Image
import json
from datetime import datetime
import re

print("✅ Libraries imported successfully!")

## Step 3: Mount Google Drive (Optional)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted at /content/drive")

## Step 4: Upload PDF File

In [ ]:
# Method A: Upload from computer
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Method B: Use from Google Drive (uncomment if using)
# pdf_path = '/content/drive/MyDrive/your-pdf-file.pdf'

print(f"✅ PDF file ready: {pdf_path}")

## Step 5: Figure Screenshot Extractor Class

In [ ]:
class FigureScreenshotExtractor:
    """
    Extracts figure screenshots from PDFs by rendering pages and cropping figure regions.
    This preserves all text, labels, and annotations within figures.
    """
    
    def __init__(self, pdf_path, output_dir="output", dpi=300, padding=20):
        """
        Initialize the figure screenshot extractor.
        
        Args:
            pdf_path (str): Path to the PDF file
            output_dir (str): Directory to save output files
            dpi (int): Resolution for page rendering (default: 300 for high quality)
            padding (int): Extra pixels to include around figure area (default: 20)
        """
        self.pdf_path = pdf_path
        self.output_dir = output_dir
        self.images_dir = os.path.join(output_dir, "images")
        self.dpi = dpi
        self.padding = padding
        
        # Create output directories
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.images_dir, exist_ok=True)
        
        # Open PDF
        self.doc = fitz.open(pdf_path)
        self.figure_counter = 0
        self.skipped_counter = 0
        
    def clean_text(self, text):
        """Clean and normalize extracted text."""
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def find_figure_regions(self, page):
        """
        Find all figure regions on a page by looking for FIGURE captions,
        then finding all content above the caption.
        Returns bounding boxes that include the complete figure and its caption.
        
        Args:
            page: PyMuPDF page object
            
        Returns:
            list: List of dictionaries with figure info and bounding boxes
        """
        blocks = page.get_text("dict")["blocks"]
        figure_regions = []
        
        # Get all images on the page with their positions
        image_list = page.get_images(full=True)
        image_bboxes = []
        for img in image_list:
            xref = img[0]
            try:
                img_rects = page.get_image_rects(xref)
                if img_rects:
                    image_bboxes.extend(img_rects)
            except:
                pass
        
        # Also detect image blocks directly (for full-bleed/decorative images)
        for block in blocks:
            if block["type"] == 1:  # Image block
                image_bboxes.append(block["bbox"])
        
        # Find all FIGURE captions
        for block in blocks:
            if block["type"] == 0:  # Text block
                # Extract text from block
                text = ""
                for line in block.get("lines", []):
                    for span in line.get("spans", []):
                        text += span.get("text", "") + " "
                
                text = self.clean_text(text)
                
                # Look for FIGURE pattern
                figure_patterns = [
                    r'FIGURE\s+([\d]+[-\.][\d]+[A-Z]?)',  # FIGURE 1-1, FIGURE 2.3
                    r'FIGURE\s+([\d]+)',                   # FIGURE 1
                    r'Fig\.?\s+([\d]+[-\.][\d]+[A-Z]?)',  # Fig. 1-1
                    r'Fig\.?\s+([\d]+)'                    # Fig 1
                ]
                
                figure_match = None
                for pattern in figure_patterns:
                    match = re.search(pattern, text, re.IGNORECASE)
                    if match:
                        figure_match = match
                        break
                
                if figure_match:
                    figure_num = figure_match.group(1)
                    caption_bbox = block["bbox"]
                    
                    # Extend caption area to include multi-line captions
                    caption_start_y = caption_bbox[1]
                    caption_end_y = caption_bbox[3]
                    caption_left_x = caption_bbox[0]
                    caption_right_x = caption_bbox[2]
                    full_caption_text = text
                    
                    # Look for continuation of caption below AND above
                    for next_block in blocks:
                        if next_block["type"] == 0:
                            next_bbox = next_block["bbox"]
                            next_text = ""
                            for line in next_block.get("lines", []):
                                for span in line.get("spans", []):
                                    next_text += span.get("text", "") + " "
                            next_text = self.clean_text(next_text)
                            
                            # Check if this block is part of the same caption (below)
                            if (next_bbox[1] >= caption_end_y and 
                                next_bbox[1] - caption_end_y < 20 and 
                                abs(next_bbox[0] - caption_left_x) < 50):
                                caption_end_y = next_bbox[3]
                                caption_right_x = max(caption_right_x, next_bbox[2])
                                full_caption_text += " " + next_text
                            
                            # Check if this block is part of the same caption (above)
                            # This handles cases where text appears before "FIGURE X-X"
                            if (next_bbox[3] <= caption_start_y and 
                                caption_start_y - next_bbox[3] < 20 and 
                                abs(next_bbox[0] - caption_left_x) < 50):
                                caption_start_y = next_bbox[1]
                                caption_left_x = min(caption_left_x, next_bbox[0])
                                caption_right_x = max(caption_right_x, next_bbox[2])
                                full_caption_text = next_text + " " + full_caption_text
                    
                    # Add extra padding below caption to ensure it's fully visible
                    # Use 3x padding below caption for better visibility
                    caption_bottom_with_padding = caption_end_y + (self.padding * 3)
                    
                    # Find all images that are ABOVE this caption
                    related_images = []
                    page_center = page.rect.width / 2
                    caption_center = (caption_left_x + caption_right_x) / 2
                    
                    for img_bbox in image_bboxes:
                        # Check if image is above the caption
                        if img_bbox[3] <= caption_start_y:  # Image bottom is at or above caption top
                            img_center = (img_bbox[0] + img_bbox[2]) / 2
                            img_width = img_bbox[2] - img_bbox[0]
                            img_height = img_bbox[3] - img_bbox[1]
                            
                            # Calculate vertical distance from image to caption
                            vertical_gap = caption_start_y - img_bbox[3]
                            
                            # Only include images that are reasonably close to the caption
                            # (within ~500 points or about 7 inches)
                            if vertical_gap > 500:
                                continue
                            
                            # Check horizontal alignment
                            horizontal_alignment = abs(img_center - caption_center)
                            
                            # Include image if:
                            # 1. Horizontally aligned with caption (within 150 points)
                            # 2. OR both are centered on page (within 100 points of page center)
                            # 3. OR image spans most of page width (full-bleed images)
                            is_aligned = horizontal_alignment < 150
                            both_centered = (abs(img_center - page_center) < 100 and 
                                           abs(caption_center - page_center) < 100)
                            is_full_width = img_width > page.rect.width * 0.6
                            
                            if is_aligned or both_centered or is_full_width:
                                related_images.append({
                                    'bbox': img_bbox,
                                    'distance': vertical_gap,
                                    'width': img_width,
                                    'height': img_height
                                })
                    
                    # Determine figure boundaries
                    if related_images:
                        # Sort by distance to find the closest images
                        related_images.sort(key=lambda x: x['distance'])
                        
                        # Use the bounding box of all related images
                        image_bboxes_only = [img['bbox'] for img in related_images]
                        min_x = min([img[0] for img in image_bboxes_only])
                        min_y = min([img[1] for img in image_bboxes_only])
                        max_x = max([img[2] for img in image_bboxes_only])
                        max_y = max([img[3] for img in image_bboxes_only])
                        
                        # Check if this looks like a full-page decorative image
                        total_height = max_y - min_y
                        total_width = max_x - min_x
                        is_large_figure = total_height > 300 or total_width > page.rect.width * 0.6
                        
                        # For multi-component figures (like FIGURE 1-5 with diagram + photo):
                        # Expand horizontally to capture labels/diagrams on the sides
                        # Look for content between the leftmost image and caption
                        figure_left_edge = min_x
                        figure_right_edge = max_x
                        
                        # Check if there's significant horizontal space on either side
                        # that might contain labels or diagrams
                        left_margin = min_x - 50  # Look 50 points to the left
                        right_margin = max_x + 50  # Look 50 points to the right
                        
                        # Scan for text/content in the figure region
                        for content_block in blocks:
                            content_bbox = content_block["bbox"]
                            # Check if block is in the vertical range of the figure
                            if (content_bbox[1] >= min_y - 20 and 
                                content_bbox[3] <= caption_start_y + 20):
                                # Expand left if there's content to the left
                                if content_bbox[0] < figure_left_edge:
                                    figure_left_edge = min(figure_left_edge, content_bbox[0])
                                # Expand right if there's content to the right  
                                if content_bbox[2] > figure_right_edge:
                                    figure_right_edge = max(figure_right_edge, content_bbox[2])
                        
                        # Use expanded boundaries
                        min_x = figure_left_edge
                        max_x = figure_right_edge
                        
                        # Adjust padding based on figure type
                        if is_large_figure:
                            # For large figures, use minimal padding to avoid capturing other content
                            x0 = max(0, min_x - self.padding)
                            y0 = max(0, min_y - self.padding)
                            x1 = min(page.rect.width, max_x + self.padding)
                            y1 = min(page.rect.height, caption_bottom_with_padding)
                        else:
                            # For normal figures, use more padding
                            x0 = max(0, min_x - self.padding * 2)
                            y0 = max(0, min_y - self.padding * 2)
                            x1 = min(page.rect.width, max_x + self.padding * 2)
                            y1 = min(page.rect.height, caption_bottom_with_padding)
                    else:
                        # No images found, use a default search area above caption
                        # This handles cases where figures might be vector graphics
                        x0 = max(0, caption_left_x - self.padding * 3)
                        y0 = max(0, caption_start_y - 450)  # Search ~450 points above
                        x1 = min(page.rect.width, caption_right_x + self.padding * 3)
                        y1 = min(page.rect.height, caption_bottom_with_padding)
                        is_large_figure = False
                    
                    figure_regions.append({
                        'figure_num': figure_num,
                        'caption': self.clean_text(full_caption_text),
                        'bbox': (x0, y0, x1, y1),
                        'caption_bbox': (caption_left_x, caption_start_y, caption_right_x, caption_end_y),
                        'num_images': len(related_images) if isinstance(related_images, list) and related_images and isinstance(related_images[0], dict) else len(related_images),
                        'is_large': is_large_figure if related_images else False
                    })
        
        return figure_regions
    
    def screenshot_figure_region(self, page, bbox, figure_num, page_num):
        """
        Take a screenshot of a specific region of the page.
        
        Args:
            page: PyMuPDF page object
            bbox: Bounding box (x0, y0, x1, y1)
            figure_num: Figure number for filename
            page_num: Page number
            
        Returns:
            str: Path to saved image
        """
        # Create a rectangle for the region
        rect = fitz.Rect(bbox)
        
        # Calculate zoom factor based on DPI
        # 72 DPI is the default, so zoom = target_dpi / 72
        zoom = self.dpi / 72
        mat = fitz.Matrix(zoom, zoom)
        
        # Render the page
        pix = page.get_pixmap(matrix=mat, clip=rect)
        
        # Generate filename
        safe_fig_num = figure_num.replace('-', '_').replace('.', '_').replace(' ', '_')
        image_filename = f"figure_{safe_fig_num}.png"
        image_path = os.path.join(self.images_dir, image_filename)
        
        # Save as PNG
        pix.save(image_path)
        
        return image_filename
    
    def extract_figures_from_page(self, page, page_num):
        """
        Extract all figure screenshots from a PDF page.
        
        Args:
            page: PyMuPDF page object
            page_num (int): Page number
            
        Returns:
            list: List of dictionaries with figure info
        """
        figure_regions = self.find_figure_regions(page)
        extracted_figures = []
        
        if not figure_regions:
            print(f"  ⚠️  No figures found on page {page_num}")
            return extracted_figures
        
        for fig in figure_regions:
            try:
                # Take screenshot of figure region
                image_filename = self.screenshot_figure_region(
                    page, 
                    fig['bbox'], 
                    fig['figure_num'],
                    page_num
                )
                
                # Calculate figure dimensions for reporting
                bbox = fig['bbox']
                width_points = bbox[2] - bbox[0]
                height_points = bbox[3] - bbox[1]
                
                # Determine figure type for reporting
                if fig['is_large']:
                    fig_type = "Large figure"
                elif fig['num_images'] > 1:
                    fig_type = "Multi-component"
                else:
                    fig_type = "Standard"
                
                figure_info = {
                    'figure_num': fig['figure_num'],
                    'page': page_num,
                    'caption': fig['caption'],
                    'image_filename': image_filename,
                    'bbox': fig['bbox'],
                    'width_points': round(width_points, 1),
                    'height_points': round(height_points, 1),
                    'num_images': fig['num_images'],
                    'type': fig_type
                }
                
                extracted_figures.append(figure_info)
                self.figure_counter += 1
                
                print(f"  ✅ Extracted FIGURE {fig['figure_num']} → {image_filename}")
                print(f"     Type: {fig_type} | Size: {round(width_points)}x{round(height_points)} points | Images: {fig['num_images']}")
                print(f"     Caption: {fig['caption'][:80]}..." if len(fig['caption']) > 80 else f"     Caption: {fig['caption']}")
                
            except Exception as e:
                print(f"  ❌ Error extracting FIGURE {fig['figure_num']}: {e}")
                self.skipped_counter += 1
        
        return extracted_figures
    
    def extract_all_figures(self, start_page=None, end_page=None):
        """
        Extract all figures from the PDF.
        
        Args:
            start_page (int, optional): First page to process (1-indexed)
            end_page (int, optional): Last page to process (1-indexed)
            
        Returns:
            dict: Extraction results with figure info and summary
        """
        print(f"\n? Processing PDF: {self.pdf_path}")
        print(f"? DPI: {self.dpi} | Padding: {self.padding}px")
        print(f"? Output directory: {self.output_dir}\n")
        
        # Determine page range
        total_pages = len(self.doc)
        start = (start_page - 1) if start_page else 0
        end = end_page if end_page else total_pages
        
        print(f"📖 Scanning pages {start + 1} to {end} of {total_pages}...\n")
        
        all_figures = []
        
        for page_num in range(start, end):
            page = self.doc[page_num]
            print(f"? Page {page_num + 1}:")
            
            figures = self.extract_figures_from_page(page, page_num + 1)
            all_figures.extend(figures)
            
            print()  # Blank line between pages
        
        # Create summary
        summary = {
            'pdf_path': self.pdf_path,
            'total_pages_scanned': end - start,
            'figures_extracted': self.figure_counter,
            'figures_skipped': self.skipped_counter,
            'output_directory': self.output_dir,
            'dpi': self.dpi,
            'padding': self.padding,
            'extraction_date': datetime.now().isoformat(),
            'figures': all_figures
        }
        
        # Save summary as JSON
        summary_path = os.path.join(self.output_dir, "extraction_summary.json")
        with open(summary_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        
        print(f"{'='*60}")
        print(f"✅ Extraction Complete!")
        print(f"{'='*60}")
        print(f"📊 Figures extracted: {self.figure_counter}")
        print(f"⚠️  Figures skipped: {self.skipped_counter}")
        print(f"📁 Output directory: {self.output_dir}")
        print(f"📄 Summary saved to: {summary_path}")
        print(f"{'='*60}\n")
        
        return summary
    
    def create_markdown_output(self, summary):
        """
        Create a markdown file with all extracted figures.
        
        Args:
            summary (dict): Extraction summary from extract_all_figures()
        """
        md_path = os.path.join(self.output_dir, "extracted_figures.md")
        
        with open(md_path, 'w', encoding='utf-8') as f:
            f.write(f"# Extracted Figures from PDF\n\n")
            f.write(f"**Source:** `{summary['pdf_path']}`  \n")
            f.write(f"**Extraction Date:** {summary['extraction_date']}  \n")
            f.write(f"**Total Figures:** {summary['figures_extracted']}  \n")
            f.write(f"**Settings:** DPI={summary['dpi']}, Padding={summary['padding']}px  \n\n")
            f.write(f"---\n\n")
            
            # Group figures by page
            figures_by_page = {}
            for fig in summary['figures']:
                page = fig['page']
                if page not in figures_by_page:
                    figures_by_page[page] = []
                figures_by_page[page].append(fig)
            
            # Write figures
            for page in sorted(figures_by_page.keys()):
                f.write(f"## Page {page}\n\n")
                
                for fig in figures_by_page[page]:
                    f.write(f"### FIGURE {fig['figure_num']}\n\n")
                    f.write(f"![FIGURE {fig['figure_num']}](images/{fig['image_filename']})\n\n")
                    f.write(f"**Caption:** {fig['caption']}\n\n")
                    f.write(f"**Details:**\n")
                    f.write(f"- Type: {fig['type']}\n")
                    f.write(f"- Size: {fig['width_points']} x {fig['height_points']} points\n")
                    f.write(f"- Image components: {fig['num_images']}\n\n")
                    f.write(f"---\n\n")
        
        print(f"📝 Markdown file created: {md_path}\n")
        return md_path


## Step 6: Configure Extraction Settings

Adjust these settings to control screenshot quality and region detection:

In [ ]:
# ===== EXTRACTION SETTINGS =====

# Output directory
output_directory = "figure_screenshots"

# Resolution for screenshots (DPI)
# Higher = better quality but larger files
#   - 150 DPI = Good for web/screen viewing
#   - 300 DPI = High quality, recommended for printing
#   - 600 DPI = Very high quality (large files)
DPI = 300

# Padding around figure region (in pixels)
# Extra space to include around detected figure area
PADDING = 30

print("✅ Extraction settings configured!")
print(f"   📁 Output: {output_directory}")
print(f"   🖼️  Resolution: {DPI} DPI")
print(f"   📏 Padding: {PADDING} pixels")
print(f"   🎯 Method: Automatic figure boundary detection")
print(f"   📊 Detects all images above FIGURE captions")

## Step 7: Run the Screenshot Extraction

In [ ]:
# Create extractor instance
extractor = FigureScreenshotExtractor(
    pdf_path, 
    output_dir=output_directory,
    dpi=DPI,
    padding=PADDING
)

# Extract all figures from the PDF
summary = extractor.extract_all_figures()

# Create markdown file with all extracted figures
markdown_file = extractor.create_markdown_output(summary)

print("\n" + "="*60)
print("📊 EXTRACTION SUMMARY")
print("="*60)
print(f"  📄 PDF: {summary['pdf_path']}")
print(f"  📖 Pages scanned: {summary['total_pages_scanned']}")
print(f"  ✅ Figures extracted: {summary['figures_extracted']}")
print(f"  ⚠️  Figures skipped: {summary['figures_skipped']}")
print(f"  📁 Output directory: {summary['output_directory']}")
print(f"  🖼️  Resolution: {summary['dpi']} DPI")
print(f"  📏 Padding: {summary['padding']} pixels")
print(f"  📝 Markdown file: {markdown_file}")
print("="*60)

## Step 8: Preview Extracted Figure Screenshots

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

# Get all extracted figures
figure_files = sorted(glob.glob(os.path.join(output_directory, "images", "figure_*.png")))

if figure_files:
    print(f"📷 Found {len(figure_files)} extracted figure screenshots\n")
    
    # List all figures
    for i, fig_path in enumerate(figure_files, 1):
        img = Image.open(fig_path)
        print(f"  {i}. {os.path.basename(fig_path)} - Size: {img.size[0]}x{img.size[1]} pixels")
    
    # Display first 6 figures
    num_to_display = min(6, len(figure_files))
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(figure_files[:num_to_display]):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(os.path.basename(img_path), fontsize=10, fontweight='bold')
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(num_to_display, 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Displayed {num_to_display} of {len(figure_files)} figure screenshots")
    print(f"\n💡 These screenshots include the complete figure with all text labels and annotations!")
else:
    print("⚠️ No figures were extracted. Try adjusting SEARCH_DISTANCE.")

## Step 9: Compare Screenshot vs Embedded Image

Let's see the difference in quality and content!

In [ ]:
if figure_files:
    # Show first figure in detail
    first_figure = Image.open(figure_files[0])
    
    plt.figure(figsize=(12, 8))
    plt.imshow(first_figure)
    plt.title(f"Sample Figure Screenshot: {os.path.basename(figure_files[0])}\n"
              f"Resolution: {DPI} DPI | Size: {first_figure.size[0]}x{first_figure.size[1]} pixels",
              fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print("\n✅ This screenshot includes:")
    print("   • The complete diagram/image")
    print("   • All text labels and annotations within the figure")
    print("   • The figure caption below")
    print("   • Proper spacing and layout as it appears in the book")
    print("\n📝 Perfect for studying medical diagrams, anatomy, or any annotated figures!")

## Step 10: Preview the Markdown File

In [ ]:
# Read and display portion of the markdown file
with open(markdown_file, "r", encoding="utf-8") as f:
    content = f.read()

print("📝 MARKDOWN PREVIEW:")
print("="*70)
print(content[:3000])  # Show first 3000 characters
print("\n...\n")
print(f"(Full document: {len(content)} characters, {content.count('FIGURE')} figure references)")

## Step 11: Download Results

Download the markdown file and figure screenshots as a zip file.

In [ ]:
import shutil

# Create a zip file with all results
zip_filename = "figure_screenshots"
shutil.make_archive(zip_filename, 'zip', output_directory)

file_size_mb = os.path.getsize(f"{zip_filename}.zip") / (1024 * 1024)

print(f"📦 Created: {zip_filename}.zip")
print(f"   Size: {file_size_mb:.2f} MB")
print(f"   Contains: Markdown file + {summary['figures_extracted']} figure screenshots at {DPI} DPI")

# Download the zip file
from google.colab import files
files.download(f"{zip_filename}.zip")

print("\n✅ Download started! Check your browser's download folder.")

## 🎯 Why Screenshots Are Better

### **Traditional Image Extraction:**
- ❌ Only gets the embedded image file
- ❌ Loses text labels overlaid on the image
- ❌ Misses annotations and callouts
- ❌ Caption not included with image

### **Screenshot Method (This Notebook):**
- ✅ Captures the entire figure as rendered
- ✅ Includes all text labels and annotations
- ✅ Preserves layout and spacing
- ✅ Includes caption in the same screenshot
- ✅ High resolution (configurable DPI)
- ✅ Perfect for medical textbooks, anatomy diagrams, etc.

### **Perfect For:**
- 📚 Medical and anatomy textbooks
- 🧬 Biology diagrams with labels
- 🔬 Scientific figures with annotations
- 📊 Charts and graphs with text
- 🎨 Any figures where text is part of the image

## ? Troubleshooting & Tips

The extractor now handles **complex figures** with multiple components:

**FIGURE 1-4 (Histology):**
- ✅ Complete labeled diagram
- ✅ Full caption: "Histology of the skin, hair, and glands"
- ✅ Caption fully visible at bottom of screenshot

**FIGURE 1-5 (Skin Layers):**
- ✅ Labeled diagram on the left (Epidermis, Dermis layers)
- ✅ Microscope photo on the right
- ✅ System detects both components and captures the complete figure

**FIGURE 1-8 (Skin Penetration):**
- ✅ All 3 diagram panels showing penetration routes
- ✅ Complete caption visible
- ✅ Full description about follicle wall, sebaceous glands, routes

**FIGURE 1-9 (Occlusive Mask):**
- ✅ Multiple photos showing application process
- ✅ All panels captured together
- ✅ Complete step-by-step visualization

### How It Works:

1. **Finds FIGURE captions** (looks for "FIGURE X-Y" pattern)
2. **Extends caption** both above and below to capture complete text
3. **Finds images** above the FIGURE caption
4. **Scans horizontally** for labels, text, and diagrams
5. **Expands boundaries** to include all related content
6. **Adds 3x padding** below caption to ensure full visibility
7. **Takes full screenshot** of the complete figure region

### If captions are cut off:

- **Increase `PADDING`** (try 30-40) for more space below caption
- System uses 3x padding below caption by default (60 pixels at padding=20)
- Check console output: Caption text should show complete description
- For very long captions, increase padding to 40 or 50

### If figures are still cut off or incomplete:

- **Increase `PADDING`** (try 40-50) for more margin
- Check console output: Shows how many images detected
- System now auto-expands to capture side labels

### If too much content is captured:

- **Decrease `PADDING`** (try 15-20) for tighter cropping
- Large figures automatically use minimal padding
- System avoids content >500 points from caption

### If no figures are found:

- Check if PDF uses "FIGURE", "Fig.", or "Fig" in captions
- Verify PDF is text-based (not scanned requiring OCR)
- Review console output for detection details

### For better quality:

- **Increase `DPI`** (300 = good, 600 = excellent)
- Higher DPI = sharper text and diagrams
- Medical/anatomy books benefit from 300+ DPI

### For smaller file sizes:

- **Decrease `DPI`** (150 = acceptable for screen)
- Lower DPI = smaller files, slightly less crisp

### Understanding the Detection:

1. Finds FIGURE captions first
2. **NEW:** Extends caption both above and below (multi-line support)
3. Locates all images above caption (within 500 points)
4. **NEW:** Scans for labels/diagrams on left/right sides
5. **NEW:** Expands to capture multi-component layouts
6. Calculates bounding box for complete figure + caption
7. **NEW:** Adds 3x padding below caption for full visibility
8. Takes high-resolution screenshot

### Figure Types Handled:

✅ **Single image** (FIGURE 1-2: DNA helix)  
✅ **Multi-panel** (FIGURE 1-3: 6 cell division phases)  
✅ **Diagram + Photo** (FIGURE 1-5: Labels + microscope image)  
✅ **Multi-panel diagrams** (FIGURE 1-8: 3 penetration route diagrams)  
✅ **Photo sequence** (FIGURE 1-9: Multiple application photos)  
✅ **Full-page decorative** images  
✅ **Complete captions** at bottom of every figure
